
# 🛡️ Fake Logo Detection — Jupyter + Streamlit 

This notebook is a **complete, local‑only pipeline** to detect **Fake vs Real logos** using **PyTorch** with **transfer learning**, plus **explainability (Grad‑CAM)** and strong **evaluation** (ROC/PR/Calibration, confusion matrix, error gallery).  
It also ships with a **Streamlit UI** 


## 1) Setup (install & imports)

In [ ]:
# If needed, uncomment to install locally in your environment
# !pip install torch torchvision torchaudio pillow numpy scikit-learn matplotlib opencv-python tqdm

import os, random, csv, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import cv2

print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())


Torch 2.8.0+cpu | CUDA available: False


## 2) Utilities

In [9]:

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def device_str():
    return 'cuda' if torch.cuda.is_available() else 'cpu'

def ensure_dir(p: str | Path):
    Path(p).mkdir(parents=True, exist_ok=True)

def save_training_log(rows: list[dict], out_csv: str):
    ensure_dir(Path(out_csv).parent)
    if not rows:
        return
    with open(out_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

def plot_roc(y_true, y_prob, out_path: str):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
    plt.plot([0,1],[0,1],'--')
    plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curve'); plt.legend(loc='lower right')
    ensure_dir(Path(out_path).parent)
    plt.savefig(out_path, dpi=200); plt.close()

def plot_pr(y_true, y_prob, out_path: str):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    plt.figure()
    plt.plot(recall, precision)
    plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision‑Recall Curve')
    ensure_dir(Path(out_path).parent)
    plt.savefig(out_path, dpi=200); plt.close()

def plot_reliability(y_true, y_prob, out_path: str, n_bins: int = 10):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy='uniform')
    plt.figure()
    plt.plot(prob_pred, prob_true, marker='o')
    plt.plot([0,1],[0,1],'--')
    plt.xlabel('Mean predicted probability'); plt.ylabel('Fraction of positives'); plt.title('Reliability (Calibration) Curve')
    ensure_dir(Path(out_path).parent)
    plt.savefig(out_path, dpi=200); plt.close()


## 3) Data loaders

In [ ]:

def make_dataloaders(data_dir: str, img_size: int = 224, batch_size: int = 32):
    normalize = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    train_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2,0.2,0.2,0.1),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        normalize
    ])
    eval_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        normalize
    ])

    train_ds = datasets.ImageFolder('D:\\M.Sc. Data Analytics\\SY\Semester 4\\DL\\CA2\\data(fake logo detection)\\train', transform=train_tfm)
    test_ds  = datasets.ImageFolder('D:\\M.Sc. Data Analytics\\SY\\Semester 4\\DL\\CA2\\data(fake logo detection)\\test',  transform=eval_tfm)
    val_ds   = datasets.ImageFolder('D:\\M.Sc. Data Analytics\\SY\\Semester 4\\DL\\CA2\\data(fake logo detection)\\test',   transform=eval_tfm)


    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    return train_dl, val_dl, test_dl, train_ds.classes


## 4) Model: Transfer Learning (ResNet‑18)

In [11]:

def build_model(num_classes: int = 2, pretrained: bool = True):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
    in_feats = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_feats, num_classes))
    return model


## 5) Train & Validate

In [ ]:

def run_epoch(model, loader, criterion, optimizer, device, train=True):
    if train: model.train()
    else: model.eval()
    loss_sum = correct = total = 0
    all_probs, all_labels = [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train: optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            if train: loss.backward(); optimizer.step()
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(1)
            loss_sum += loss.item() * x.size(0)
            correct += (preds == y).sum().item()
            total += x.size(0)
            all_probs.append(probs[:,1].detach().cpu().numpy())  # class 1 = fake
            all_labels.append(y.detach().cpu().numpy())
    all_probs = np.concatenate(all_probs) if len(all_probs)>0 else np.array([])
    all_labels = np.concatenate(all_labels) if len(all_labels)>0 else np.array([])
    return loss_sum/total, correct/total, all_probs, all_labels

def train_single_split(data_dir="D:\\M.Sc. Data Analytics\\SY\\Semester 4\\DL\\CA2\\data(fake logo detection)", img_size=224, batch_size=32, lr=3e-4, epochs=10, seed=42, save_path="D:\\M.Sc. Data Analytics\\SY\\Semester 4\\DL\\CA2\\models\\best_model.pt"):
    seed_everything(seed)
    device = torch.device(device_str())
    ensure_dir(Path(save_path).parent)
    train_dl, val_dl, test_dl, classes = make_dataloaders(data_dir, img_size, batch_size)
    model = build_model(num_classes=len(classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    best_state, best_val = None, 0.0
    logs = []
    for ep in range(1, epochs+1):
        tr_loss, tr_acc, _, _ = run_epoch(model, train_dl, criterion, optimizer, device, True)
        va_loss, va_acc, _, _ = run_epoch(model, val_dl, criterion, optimizer, device, False)
        logs.append({'epoch':ep,'train_loss':tr_loss,'train_acc':tr_acc,'val_loss':va_loss,'val_acc':va_acc})
        print(f'Epoch {ep:02d} | train {tr_loss:.4f}/{tr_acc:.3f} | val {va_loss:.4f}/{va_acc:.3f}')
        if va_acc > best_val:
            best_val = va_acc; best_state = model.state_dict()

    torch.save({'state_dict': best_state, 'classes': classes, 'img_size': img_size, 'arch':'resnet18'}, save_path)
    save_training_log(logs, "D:\\M.Sc. Data Analytics\\SY\\Semester 4\\DL\\CA2\\outputs\\training_log.csv")

    # Final test
    model.load_state_dict(best_state)
    te_loss, te_acc, te_prob, te_y = run_epoch(model, test_dl, criterion, optimizer, device, False)
    print(f'\nTest: loss={te_loss:.4f} acc={te_acc:.4f}')

    # Reports & plots
    y_pred = (te_prob>0.5).astype(int)
    print('\nClassification Report (test):')
    print(classification_report(te_y, y_pred, target_names=classes, digits=4))

    # Confusion Matrix
    cm = confusion_matrix(te_y, y_pred)
    plt.figure(figsize=(4,4))
    plt.imshow(cm, interpolation='nearest')
    plt.title('Confusion Matrix (Test)'); plt.colorbar()
    ticks = np.arange(len(classes)); plt.xticks(ticks, classes, rotation=45); plt.yticks(ticks, classes)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            c = 'white' if cm[i,j]>cm.max()/2 else 'black'
            plt.text(j,i,cm[i,j],ha='center',va='center',color=c)
    ensure_dir('outputs'); plt.tight_layout(); plt.savefig("D:\\M.Sc. Data Analytics\\SY\\Semester 4\\DL\\CA2\\outputs\\confusion_matrix.png", dpi=200); plt.close()

    plot_roc(te_y, te_prob, "D:\\M.Sc. Data Analytics\\SY\\Semester 4\\DL\\CA2\\outputs\\curves\\roc.png")
    plot_pr(te_y, te_prob, "D:\\M.Sc. Data Analytics\\SY\\Semester 4\\DL\\CA2\\outputs\\curves\\pr.png")
    plot_reliability(te_y, te_prob, "D:\\M.Sc. Data Analytics\\SY\\Semester 4\\DL\\CA2\\outputs\\curves\\reliability.png")

    return save_path, classes


## 6) Grad‑CAM Explainability

In [13]:

def grad_cam(model, img_tensor, target_layer_name='layer4'):
    activations, gradients = {}, {}
    def fwd_hook(_, __, out): activations['value'] = out.detach()
    def bwd_hook(_, __, grad_out): gradients['value'] = grad_out[0].detach()

    layer = dict([*model.named_modules()])[target_layer_name]
    fh = layer.register_forward_hook(fwd_hook)
    bh = layer.register_full_backward_hook(bwd_hook)

    model.zero_grad()
    output = model(img_tensor)
    prob = torch.softmax(output, dim=1)[0]
    pred_idx = int(output.argmax(1).item())
    score = output[0, pred_idx]
    score.backward()

    acts = activations['value'][0]  # [C,H,W]
    grads = gradients['value'][0]   # [C,H,W]
    weights = grads.mean(dim=(1, 2))
    cam = (weights[:, None, None] * acts).sum(dim=0)
    cam = torch.relu(cam)
    cam = (cam - cam.min()) / (cam.max() + 1e-6)
    fh.remove(); bh.remove()
    return cam.cpu().numpy(), pred_idx, float(prob[pred_idx].item())

def overlay_cam_on_image(pil_img, cam):
    import numpy as np, cv2
    img_np = np.array(pil_img.convert('RGB'))
    h, w, _ = img_np.shape
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
    overlay = (0.35 * heatmap + 0.65 * img_np).astype(np.uint8)
    return overlay


## 7) Run training (single split)

In [ ]:

# Edit these paths if needed, then run this cell.

_ = ensure_dir('models'); _ = ensure_dir('outputs')

ckpt_path, classes = train_single_split(
    data_dir="D:\\M.Sc. Data Analytics\\SY\\Semester 4\\DL\\CA2\\data(fake logo detection)",
    img_size=224,
    batch_size=32,
    lr=3e-4,
    epochs=10,
    seed=42,
    save_path="D:\\M.Sc. Data Analytics\\SY\\Semester 4\\DL\\CA2\\models\\best_model.pt"
)
print('Saved model ->', ckpt_path)
print('Classes:', classes)


c:\Users\Shondita\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 01 | train 0.4906/0.775 | val 0.2577/0.903
Epoch 02 | train 0.2197/0.903 | val 0.8412/0.861
Epoch 03 | train 0.1758/0.936 | val 0.1540/0.933
Epoch 04 | train 0.1345/0.942 | val 0.1890/0.939
Epoch 05 | train 0.1458/0.948 | val 0.2725/0.867
Epoch 06 | train 0.1775/0.930 | val 0.8283/0.806
Epoch 07 | train 0.1379/0.951 | val 0.1348/0.927
Epoch 08 | train 0.1302/0.956 | val 0.1741/0.921
Epoch 09 | train 0.0923/0.965 | val 0.1132/0.939
Epoch 10 | train 0.0860/0.968 | val 0.3070/0.909

Test: loss=0.3070 acc=0.9091

Classification Report (test):
              precision    recall  f1-score   support

        Fake     0.8800    1.0000    0.9362       110
     Genuine     1.0000    0.7273    0.8421        55

    accuracy                         0.9091       165
   macro avg     0.9400    0.8636    0.8891       165
weighted avg     0.9200    0.9091    0.9048       165

Saved model -> D:\M.Sc. Data Analytics\SY\Semester 3\Project\models\best_model.pt
Classes: ['Fake', 'Genuine']


## 8) Inference & Grad‑CAM demo on a single image

In [15]:

def load_model_for_inference(ckpt_path='models/best_model.pt'):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    classes = ckpt['classes']; img_size = ckpt.get('img_size', 224)
    model = models.resnet18(weights=None)
    in_feats = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_feats, len(classes)))
    model.load_state_dict(ckpt['state_dict']); model.eval()
    tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    return model, tfm, classes
